Fetch Models

In [ ]:
import tensorflow as tf

mobilenet_model = tf.keras.models.load_model('/../Optimized_MobileNetV2_225input_model.h5')
resnet_model = tf.keras.models.load_model('/../ResNet50V2_model.h5')
inception_model = tf.keras.models.load_model('/../Optimized_InceptionV3_225input_model.h5')


Data Preprocessing

In [ ]:
import os
class_labels = sorted(os.listdir('/../split_dataset/train'))
print(class_labels)

import numpy as np
from tensorflow.keras.preprocessing import image

# Update this list based on your class names
class_labels = class_labels;   # Replace with actual class names

def preprocess_img(img_path):
    img = image.load_img(img_path, target_size=(225, 225))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


PIL Preprocessing & Sample UI Creation

In [ ]:
from PIL import Image
import numpy as np
import tensorflow as tf

# Preprocessing function
def preprocess_pil_image(img, target_size):
    img_resized = img.resize(target_size)
    img_array = np.array(img_resized).astype(np.float32)

    # Ensure 3 channels
    if img_array.shape[-1] != 3:
        img_array = np.stack((img_array,) * 3, axis=-1)

    return img_array

# Symptom descriptions
symptoms = {
    'boron': "Young leaves are distorted or brittle. Fruit development is poor.",
    'calcium': "Young leaves curl, tip burn occurs, and roots are underdeveloped.",
    'healthy': "Leaves are green, firm, and well-formed with no visible issues.",
    'iron': "Yellowing starts between veins on young leaves (interveinal chlorosis).",
    'magnesium': "Yellowing starts from the leaf edges inward, mainly on older leaves.",
    'manganese': "Yellowing with green veins, brown spots on young leaves.",
    'potassium': "Leaf margins dry and curl; yellowing on older leaves first.",
    'sulphur': "Uniform yellowing of young leaves, stunted growth.",
    'zinc': "Shortened internodes, small leaves, and interveinal chlorosis."
}

# Recommended actions
recommended_actions = {
    'boron': "Apply borax or boric acid as foliar spray (0.1%). Maintain soil pH 5.5–6.5.",
    'calcium': "Apply calcium nitrate or gypsum. Use 0.5% foliar spray for quick correction.",
    'healthy': "No action needed. Maintain current practices and monitor regularly.",
    'iron': "Spray iron chelates (Fe-EDTA) or ferrous sulfate. Avoid over-liming.",
    'magnesium': "Apply Epsom salt (1–2% foliar spray). Avoid excess potassium.",
    'manganese': "Apply manganese sulfate (0.5% foliar spray). Avoid over-liming.",
    'potassium': "Apply KCl or K₂SO₄. Split application for better uptake.",
    'sulphur': "Apply ammonium sulfate or elemental sulfur. Spray 1% potassium sulfate.",
    'zinc': "Spray zinc sulfate (0.5%). Broadcast with organic manure if needed."
}

# Prediction function
def predict_ensemble(img):
    img = img.convert('RGB')

    # Preprocess image
    mobilenet_array = preprocess_pil_image(img, (225, 225))
    resnet_array = preprocess_pil_image(img, (224, 224))
    inception_array = preprocess_pil_image(img, (225, 225))

    mobilenet_array = tf.keras.applications.mobilenet_v2.preprocess_input(mobilenet_array)
    resnet_array = tf.keras.applications.resnet_v2.preprocess_input(resnet_array)
    inception_array = tf.keras.applications.inception_v3.preprocess_input(inception_array)

    mobilenet_array = np.expand_dims(mobilenet_array, axis=0)
    resnet_array = np.expand_dims(resnet_array, axis=0)
    inception_array = np.expand_dims(inception_array, axis=0)

    # Predictions
    pred1 = mobilenet_model.predict(mobilenet_array, verbose=0)[0]
    pred2 = resnet_model.predict(resnet_array, verbose=0)[0]
    pred3 = inception_model.predict(inception_array, verbose=0)[0]

    # Weighted average
    weights = [0.4, 0.3, 0.3]
    final_pred = pred1 * weights[0] + pred2 * weights[1] + pred3 * weights[2]

    # Output
    class_index = np.argmax(final_pred)
    confidence = final_pred[class_index] * 100
    predicted_class = class_labels[class_index].lower()

    # Build message
    if predicted_class == "healthy":
        result_text = f"The banana leaf is healthy.  Confidence of prediction:({confidence:.2f}%)"
    else:
        result_text = f"The banana leaf likely lacks **{class_labels[class_index]}**.  Confidence of prediction: ({confidence:.2f}%)"

    return (
        f"{result_text}\n\n"
        f"Symptoms: {symptoms[predicted_class]}\n"
        f"Recommended Action: {recommended_actions[predicted_class]}"
    )


Gradio Interface

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=predict_ensemble,
    inputs=gr.Image(type="pil"),
    outputs=gr.Textbox(),
    title="Banana Leaf Nutrient Deficiency Detector",
    description=("Upload an image of a banana leaf to detect potential nutrient deficiencies using an ensemble of deep learning models. "
        "The system will identify the nutrient issue (if any), explain its symptoms, and suggest corrective actions to help improve crop health.")
)

interface.launch()
